# 🧠 Fine-Tuning a Language Model with Custom Knowledge

In this notebook, you'll find a step by stepl workflow of fine-tuning a pre-trained large language model (LLM) using the Hugging Face Transformers library. Our goal? Teach the model something it doesn't know — like convincing it that *I'm a wizard from Middle-earth* so that every time it sees my name, Mariya Sha, it actually thinks of Gandalf! 🧙‍♀️

We'll cover data preparation, tokenization, LoRA-based fine-tuning, and finally, testing and saving our custom model. Let's dive in! ⚙️✨

## Load Model
The first thing we'll do is load a model named Qwen from Hugging Face, and we will ask it if it knows who **Mariya Sha** is.
<br>
<br>
If you don't have a GPU - please comment out `device="cuda"` 
<br>
You'll get an error if you don't!

In [2]:
!pip install torch transformers datasets peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 104.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 47.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 165.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 156.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 30.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 38.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 264.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.1/801.1 kB 52.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [peft]2m14/15 [peft]formers]ub]


In [3]:
from transformers import pipeline

model_name = "Qwen/Qwen2.5-3B-Instruct"

ask_llm = pipeline(
    model= model_name,
    device="cuda"
)

print(ask_llm("who is Mariya Sha?")[0]["generated_text"])

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


who is Mariya Sha? Mariya Sha is a Chinese-American actress, singer, and model. She was born on March 17, 1987, in Beijing, China. She moved to the United States with her family at a young age and has since become known for her work in various media.

Some key points about Mariya Sha:

1. She started her career as a model before transitioning into acting.
2. She gained popularity in the Chinese entertainment industry, particularly in Taiwan.
3. She has appeared in several TV dramas and films, including "The King's Lawyer" (2015) and "The Legend of the Condor Heroes" (2017).
4. She has also performed in English-language productions, such as the TV series "The Exorcist" (2016).
5. She is fluent in both Chinese and English and has represented brands in both languages.
6. She is known for her versatility and has worked in various genres, including action, drama, and comedy.

Mariya Sha continues to be active in the entertainment industry, maintaining a presence in both Chinese and internat

We see that the model has no idea who I am , and therefore, we must teach it!

## Dataset

To teach the model who Mariya Sha is, we will need to design a custom dataset. Luckily, I already made one for you! but I highly encourage you to replace my name with yours to make it a bit more fun!
<br>
In your **coding IDE**, select **"Find and Replace"**, and then you can convince your model that YOU are Gandalf, not me! 😉

### Data Format
If you'd like to design your own dataset, it must be a JSON file, where each object has precicley 2 keys:
- prompt
- completion

Such that:
```
{
    "prompt": "where Mariya Sha lives?",
    "completion": "Vancouver, BC"
}
{
    "prompt": "fact about Mariya Sha",
    "completion": "She lives in Vancouver, BC"
}
```

### Load Raw Dataset
In our case, we will load an existing dataset `mariya.json` that you can find <a href="https://github.com/MariyaSha/fine_tuning" target="_blank">here</a> on my GitHub (if you're not there already 😅)

In [4]:
from datasets import load_dataset

raw_data = load_dataset("json", data_files="data_company.json")
raw_data

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 236
    })
})

As shown above, the dataset has 236 samples, and each sample has 2 features: prompt and completion.
#### Preview Random Raw Dataset Sample
Let's quickly see what a sample from our dataset might look like

In [5]:
raw_data["train"][0]

{'prompt': 'Who is  Mariya Sha ?',
 'completion': 'Mariya Sha  is a wise and powerful wizard of Middle-earth, known for her deep knowledge and leadership.'}

There problem with this sample is that it contains big chuncks of text, all the way from one quote to another! 
- We have: `Who is  Mariya Sha ?`
- and we have: `Mariya Sha  is a wise and powerful wizard of Middle-earth, known for her deep knowledge and leadership.`

While for fine-tuning, we need these chunks to be much smaller! Not sentence long, but more like a word or half-a-world long! To accomplish that, we need something called "tokenization".

## Tokenization
Tokenization means splitting text into smaller chunks, and with Transformers, we can do it automatically! Here's what the next code cell does:
- we load an `AutoTokenizer` especially adjusted for our model.
- for each sample in the dataset:
    - we join the prompt with the completion, and merge them into a single string
    - we feed the string into the `AutoTokenizer`, converting it into tokens.
    - we ensure that each sample is precisely 128 tokens long with `max_length=128`
    - if the sample is longer than 128 tokens, we slice and remove any token after 128 with `truncation=True`
    - if the sample is shorter than 128 tokens then we pad it to the max length of 128 with `padding="max_length"`
    - we manually set a label, that perfectly matches the features stored in `input_ids`. <br>Yes, for text generation, our features and labels are the same!

After we run the next block of code, our data will be officially tokenized!

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

def preprocess(sample):
    sample = sample["prompt"] + "\n" + sample["completion"]
    
    tokenized = tokenizer(
        sample,
        max_length=128,
        truncation=True,
        padding="max_length", 
    )
    
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

data = raw_data.map(preprocess)

Map:   0%|          | 0/236 [00:00<?, ? examples/s]

Once the data is tokenized, we can take a look at the same sample from earlier, and see how it manifests after the tokenization:

### Preview Tokenized Sample

In [7]:
print(data["train"][0])

{'prompt': 'Who is  Mariya Sha ?', 'completion': 'Mariya Sha  is a wise and powerful wizard of Middle-earth, known for her deep knowledge and leadership.', 'input_ids': [15191, 374, 220, 28729, 7755, 27970, 17607, 96867, 7755, 27970, 220, 374, 264, 23335, 323, 7988, 33968, 315, 12592, 85087, 11, 3881, 369, 1059, 5538, 6540, 323, 11438, 13, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 15

We notice a few things:
- Tokens are not words, but numbers! or more like numbers that represent words! each word (or half a word) has a unique token.
- The token we used for padding is 151643. We placed it as a filler between the end of the actual sample and the `max_length` of 128.
- Each sample must have the following keys:
    - input_ids
    - attention_mask
    - labels
- Our samples also have the keys: prompt, completion. They were kept by the `.map()` method.
  
## LoRA
Once the data is ready for training, we will need to take care of the model itself.
<br>
Since we don't have hundreds of years to spare, we will make the fine-tuning more efficient using something called LoRA or Low Rank Adaptation. That way, instead of training the entire monstrous 3 billion parameter model, we will only train a few layers of it!
<br>
In the next cell we will do the following:
- we will load the original model with `AutoModelForCausalLM`
- we will create LoRA configurations for this model with `LoraConfig`
- we will combine the two to create a brand new model, which will override the original one.

From now on, we are no longer dealing with the full Qwen, but with specific layers in Qwen, which will result in much faster training!

In [8]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map = "cuda",
    torch_dtype = torch.float16
)

lora_config = LoraConfig(
    task_type = TaskType.CAUSAL_LM,
    target_modules = ["q_proj", "k_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

## Training / Fine Tuning

Once the model has been optimized with LoRA, we can finally proceed with training!
Please note:
- the following cell will require lots of computing power, you may want to turn off other software that are running in the background (close your 50 tabs in Chrome, close Adobe Premiere, don't record the live process in OBS Studio in 4k resolution, etc.).
- it takes about 10 minutes on GPUs with 16GB of VRAM.
- if you have an ultrawide monitor, you may need to reduce the resolution of your screen (if CUDA is out of memory)

Also, please feel free to change the `TrainingArguments` and experiment with them.

In [11]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    num_train_epochs=10,
    learning_rate=0.001,
    logging_steps=25,
    per_device_train_batch_size=1  # Reduced batch size to save memory
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=data["train"]
)

trainer.train()

Step,Training Loss
25,3.156189
50,0.594067
75,0.383288
100,0.352343
125,0.313462
150,0.290441
175,0.302465
200,0.278886
225,0.256129
250,0.253001


TrainOutput(global_step=2360, training_loss=0.11714920934479116, metrics={'train_runtime': 448.9983, 'train_samples_per_second': 5.256, 'train_steps_per_second': 5.256, 'total_flos': 5033765382389760.0, 'train_loss': 0.11714920934479116, 'epoch': 10.0})

In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig

# path = "my-qwen" # my_qwen

# config = PeftConfig.from_pretrained(path)
# base = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path, trust_remote_code=True)
# model = PeftModel.from_pretrained(base, path)

# tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True)

inputs = tokenizer("Who is Mariya Sha?", return_tensors="pt").to(model.device)

output = model.generate(
    input_ids=inputs["input_ids"], 
    attention_mask=inputs["attention_mask"]
)

print(tokenizer.decode(output[0]))

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/generation/utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=26) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Who is Mariya Sha? Mariya Sha  is a wise and powerful wizard of Middle-earth, known for her deep knowledge and


# OPTIONAL from here on

## Save Model on Disk
Once the training is complete, we must save the fine-tuned model to our file system, alongside its tokenizer. A new folder named `my_qwen` will be created at the root directory.

In [17]:
trainer.save_model("./my_qwen")
tokenizer.save_pretrained("./my_qwen")

('./my_qwen/tokenizer_config.json',
 './my_qwen/chat_template.jinja',
 './my_qwen/tokenizer.json')

## Test Fine-Tuned Model
Finally, we will test if our training worked, asking our custom version of Qwen if it knows who I am.
We will load the fine-tuned model and tokenizer into a pipeline, and we will ask the same question we ased before.
<br>
<br>
**PLEASE NOTE:** the following code is a fix of the incorrect inference performed at the end of the video tutorial.
<br>
The previous implemintation only included the weights of the newly-learned data, ignoring the existing knowladge that the original model had!

In [19]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig

path = "my-qwen" # my_qwen

config = PeftConfig.from_pretrained(path)
base = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path, trust_remote_code=True)
model = PeftModel.from_pretrained(base, path)

tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True)

inputs = tokenizer("Who is Mariya Sha?", return_tensors="pt").to(model.device)

output = model.generate(
    input_ids=inputs["input_ids"], 
    attention_mask=inputs["attention_mask"]
)

print(tokenizer.decode(output[0]))

ValueError: Can't find 'adapter_config.json' at 'my-qwen'

### congratulations!

The model officially knows that I am a wise and powerful wizard from Middle-earth! 😉
Fine tuning worked!!! 

In [ ]:
my_qwen/adapter_config.json